
# SEGMENTACIÓN DE CLIENTES CON K-MEANS (Notebook Práctico)
**Autor:** Tú 🙌  

**Fecha de generación:** 2025-08-26 17:59:14

---

## Objetivo
Agrupar **100 clientes** según dos variables:
- **Monto gastado (MXN)**  
- **Frecuencia de compra (visitas/mes)**

Usaremos **K-Means** para descubrir segmentos de clientes y evaluar la calidad de los clusters con las métricas **Inercia (SSE)** y **Coeficiente de Silueta**.

---

## Requisitos previos
Ejecuta en tu entorno (Terminal / CMD) si aún no tienes las dependencias:
```bash
pip install numpy matplotlib scikit-learn
```
> Recomendado: usar un entorno virtual (e.g., `python -m venv venv` y luego `source venv/bin/activate` o en Windows `venv\Scripts\activate`).

---

## Contenido del Notebook
1. Teoría breve de K-Means y consideraciones prácticas
2. Generación de datos simulados (100 clientes)
3. Escalamiento de datos (por qué y cómo)
4. Selección de **K** con Método del Codo y Silueta
5. Entrenamiento del modelo K-Means (K=3)
6. Interpretación de centroides en escala original
7. Visualización e interpretación de clusters
8. Etiquetado legible para stakeholders
9. Exportación de resultados (CSV) y siguientes pasos



## 1) Teoría breve de K-Means
**K-Means** es un algoritmo de aprendizaje no supervisado que:
- Divide los datos en **K** grupos minimizando la suma de distancias cuadradas de cada punto a su centroide (SSE/Inercia).
- Alterna entre:
  1. **Asignación:** cada punto se asigna al centroide más cercano.
  2. **Actualización:** se recalculan los centroides como el promedio de los puntos asignados.

### Consideraciones clave
- **Escala de variables:** K-Means es sensible a la escala; por eso **estandarizamos** (media 0, desviación 1).
- **Elección de K:** No hay un K “perfecto”. Se usa el **método del codo** (mirar el “doblez” en la gráfica de Inercia) y **silueta** (valores cercanos a 1 suelen ser mejores; < 0 sugiere mala separación).
- **Inicialización:** usamos múltiples inicializaciones (`n_init`) para evitar mínimos locales.
- **Formas de clúster:** K-Means supone clusters “compactos” y relativamente esféricos; si hay formas arbitrarias, considerar DBSCAN o aglomerativo.
- **Outliers:** pueden sesgar centroides; conviene detectarlos o usar métodos robustos según el caso.


In [ ]:

# 2) Importación de bibliotecas y configuración
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from collections import Counter

# Para reproducibilidad
np.random.seed(42)

# Mostrar versión de librerías (opcional)
import sys, sklearn
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)



## 3) Generación de datos simulados (100 clientes)
Simularemos tres segmentos realistas:
- **A:** bajo gasto / baja frecuencia  
- **B:** gasto medio / frecuencia media  
- **C (VIP):** alto gasto / alta frecuencia  

Luego concatenamos, acotamos rangos, mezclamos filas y formamos la matriz **X = [monto, frecuencia]**.


In [ ]:

# Tamaños por segmento
n_a, n_b, n_c = 40, 35, 25

# Segmento A
monto_a = np.random.normal(loc=400, scale=120, size=n_a)
freq_a  = np.random.normal(loc=3,   scale=1.2, size=n_a)

# Segmento B
monto_b = np.random.normal(loc=1200, scale=250, size=n_b)
freq_b  = np.random.normal(loc=12,   scale=3,   size=n_b)

# Segmento C (VIP)
monto_c = np.random.normal(loc=2200, scale=300, size=n_c)
freq_c  = np.random.normal(loc=24,   scale=4,   size=n_c)

# Concatenación
monto = np.concatenate([monto_a, monto_b, monto_c])
freq  = np.concatenate([freq_a,  freq_b,  freq_c])

# Acotar a rangos plausibles
monto = np.clip(monto, 100, 5000)
freq  = np.clip(freq,  1,   50)

# Matriz X
X = np.column_stack([monto, freq])

# Mezclar filas para evitar sesgos por orden
idx = np.random.permutation(X.shape[0])
X = X[idx, :]

print("Shape de X:", X.shape)
print("Primeros 5 registros (Monto MXN, Frecuencia/mes):\n", X[:5])



## 4) Escalamiento de datos
Estandarizamos cada columna (media 0, desviación 1) con **StandardScaler** para que **monto** y **frecuencia** tengan el mismo peso.


In [ ]:

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Media post-escalamiento (≈0):", np.round(X_scaled.mean(axis=0), 4))
print("Desv. estándar post-escalamiento (≈1):", np.round(X_scaled.std(axis=0, ddof=0), 4))



## 5) Selección de **K** (Codo y Silueta)
Evaluamos **K = 2..7** y graficamos:
- **Inercia (SSE)** → buscar el “codo”
- **Silueta** → mayor suele ser mejor (rango -1 a 1)


In [ ]:

ks = range(2, 8)
inertias = []
silhouettes = []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

# Método del codo
plt.figure(figsize=(5, 4))
plt.plot(list(ks), inertias, marker='o')  # No especificamos colores (recomendación del entorno)
plt.title("Método del Codo para Selección de K")
plt.xlabel("Número de Clusters (K)")
plt.ylabel("Inercia (SSE)")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

# Coeficiente de Silueta
plt.figure(figsize=(5, 4))
plt.plot(list(ks), silhouettes, marker='o')
plt.title("Coeficiente de Silueta para Selección de K")
plt.xlabel("Número de Clusters (K)")
plt.ylabel("Silueta")
plt.grid(True, linestyle="--", alpha=0.4)
plt.show()

best_k_by_sil = ks[np.argmax(silhouettes)]
print("K con mayor silueta:", best_k_by_sil)



## 6) Entrenamiento K-Means con **K = 3**
Elegimos **K=3** (consistente con la simulación) y reportamos métricas clave.


In [ ]:

k = 3
kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
kmeans.fit(X_scaled)

labels = kmeans.labels_
centroids_scaled = kmeans.cluster_centers_
sse = kmeans.inertia_
sil = silhouette_score(X_scaled, labels)

print(f"Número de Clusters: {k}")
print(f"Inercia (SSE): {sse:.2f}")
print(f"Coeficiente de Silueta: {sil:.3f}")
print("Distribución de Clientes por Cluster:", dict(Counter(labels)))



## 7) Centroides en escala original
Convertimos los centroides **a la escala original** (MXN y visitas/mes) para interpretarlos de forma intuitiva.


In [ ]:

centroids_original = scaler.inverse_transform(centroids_scaled)

for i, c in enumerate(centroids_original):
    print(f"Centroide {i}: Monto ≈ {c[0]:.0f} MXN | Frecuencia ≈ {c[1]:.1f} visitas/mes")



## 8) Visualización de los clusters
Gráfico de dispersión **(Monto vs Frecuencia)** con centroides marcados con **X**.  
> Nota: No fijamos colores específicos para cumplir buenas prácticas del entorno.


In [ ]:

plt.figure(figsize=(6, 5))

for i in range(k):
    plt.scatter(
        X[labels == i, 0], 
        X[labels == i, 1], 
        s=35, alpha=0.8, label=f"Cluster {i}"
    )

# Centroides
plt.scatter(
    centroids_original[:, 0],
    centroids_original[:, 1],
    marker="X", s=200, label="Centroides"
)

plt.title("Segmentación de Clientes con K-Means")
plt.xlabel("Monto Gastado (MXN)")
plt.ylabel("Frecuencia de Compra (visitas/mes)")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.35)
plt.show()



## 9) Interpretación y etiquetado legible
Ordenamos clusters por **monto promedio** y asignamos etiquetas descriptivas para stakeholders.


In [ ]:

orden = np.argsort(centroids_original[:, 0])
mapa_etiquetas = {
    orden[0]: "Bajo gasto / baja-moderada frecuencia",
    orden[1]: "Gasto medio / frecuencia media",
    orden[2]: "Alto gasto / alta frecuencia"
}

print("Interpretación sugerida por centroide:")
for i in range(k):
    c = centroids_original[i]
    print(f"- Cluster {i} → {mapa_etiquetas[i]} (Monto ≈ {c[0]:.0f} MXN, Frecuencia ≈ {c[1]:.1f} visitas/mes)")

# Mostrar primeros 10 clientes
print("\nPrimeros 10 clientes (Monto, Frecuencia, Cluster):")
for i in range(10):
    print(f"Cliente {i:02d} -> (Monto = {X[i,0]:.0f} MXN, Frecuencia = {X[i,1]:.0f} visitas/mes) | {mapa_etiquetas[labels[i]]}")



## 10) Exportación de resultados (CSV) y siguientes pasos
Guardamos los datos con sus etiquetas para análisis adicional (e.g., segmentación RFM, campañas de marketing).

**Siguientes pasos sugeridos:**
- Probar **K** alternativos y comparar métricas/negocio.
- Añadir más características (e.g., ticket promedio, tiempo desde última compra).
- Evaluar métodos alternativos: **K-Medoids**, **DBSCAN**, **Aglomerativo**.
- Detectar outliers y evaluar su impacto.


In [ ]:

import pandas as pd
from pathlib import Path

df = pd.DataFrame(X, columns=["monto_mxn", "frecuencia_mes"])
df["cluster"] = labels
df["segmento"] = [mapa_etiquetas[l] for l in labels]

out_dir = Path("/mnt/data/segmentacion_clientes_kmeans")
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "clientes_segmentados.csv"
df.to_csv(csv_path, index=False, encoding="utf-8")

# También exportamos centroides en escala original
cent_df = pd.DataFrame(centroids_original, columns=["monto_mxn", "frecuencia_mes"])
cent_df["cluster"] = range(k)
cent_path = out_dir / "centroides_original.csv"
cent_df.to_csv(cent_path, index=False, encoding="utf-8")

print("Archivos exportados:")
print(" -", csv_path)
print(" -", cent_path)

# Vista rápida
df.head(10)



---

### ¡Listo!
Ya tienes un flujo completo de **segmentación de clientes con K-Means**:
- Datos → Escalamiento → Selección de K → Entrenamiento → Interpretación → Exportación.

Si deseas, puedo **adaptar el notebook** para tus datos reales (CSV) y añadir un **informe automático** en HTML o PDF con gráficos y métricas clave.
